# gridMET Foundations: Working with Spatial Meteorological Data

Welcome! This notebook walks you through the building blocks of spatial meteorological data analysis using gridMET -- a gridded climate dataset that covers the contiguous United States at roughly 4 km resolution with daily observations from 1979 to present.

You do not need any prior experience with geographic data or maps. Each section explains the concept in plain English before showing you any code. Work through the cells from top to bottom and answer the reflection questions as you go.

**Variable we will use throughout:** Maximum daily air temperature (tmmx)

**Study area:** Louisiana

---
## Section 0: Setup and Library Imports

Before we can do anything, we need to load the Python **libraries** we will use. A library is a collection of pre-written code that gives you extra capabilities beyond what plain Python provides. Think of it like a toolbox: Python gives you a hammer and a screwdriver, but libraries give you the full workshop.

Here is what each library does:

- **xarray**: Works with labeled, multi-dimensional arrays. This is the main tool we use to open and work with gridMET NetCDF files. Think of it as pandas but for gridded data with spatial and time dimensions.
- **numpy**: The foundation of numerical computing in Python. It provides fast array operations and math functions.
- **matplotlib**: The standard Python plotting library. We use it to make all of our charts and maps.
- **cartopy**: Adds geographic map projections and features (coastlines, borders, state outlines) to matplotlib plots.
- **geopandas**: Like pandas, but for spatial data. It can read shapefiles (which store geographic boundaries like parish outlines) and lets you do spatial operations like finding which points fall inside a polygon.
- **requests**: A simple library for downloading files from the web programmatically (from inside your code, without using a browser).
- **shapely**: Provides geometric objects (points, polygons) and operations (like testing whether a point is inside a polygon). geopandas uses it under the hood.

Run the cell below to import everything.

In [1]:
%matplotlib inline
# Tell Jupyter to show plots inline (inside the notebook)

import xarray as xr          # N-dimensional labeled arrays for gridded data
import numpy as np           # Numerical arrays and math
import matplotlib.pyplot as plt  # Plotting
import matplotlib.patches as mpatches  # Used to build custom map legends
import cartopy.crs as ccrs   # Map coordinate reference systems
import cartopy.feature as cfeature  # Built-in geographic features for maps
import geopandas as gpd      # Spatial dataframes for shapefiles and geometries
import requests               # HTTP requests for downloading files
from shapely.geometry import Point  # Geometric point objects
import os                    # File system operations (e.g., checking file size)
import warnings
warnings.filterwarnings('ignore')  # Suppress minor deprecation warnings

print('All libraries loaded successfully.')
print(f'xarray version: {xr.__version__}')
print(f'geopandas version: {gpd.__version__}')

All libraries loaded successfully.
xarray version: 2026.4.0
geopandas version: 1.1.3


### Reflection Questions -- Section 0

*Double-click this cell to edit it and type your answers.*

**Q1.** What do you think is the difference between a library and a built-in Python function like `print()` or `len()`?

*Your answer:*

---

**Q2.** Why might spatial data analysis require specialized libraries like cartopy and geopandas rather than just pandas?

*Your answer:*

---
## Section 1: Downloading gridMET Data Programmatically

### What is gridMET?

gridMET is a gridded surface meteorological dataset that covers the contiguous United States. It provides daily estimates of climate variables -- like maximum temperature, precipitation, wind speed, and humidity -- on a regular grid of cells roughly 4 km x 4 km in size. The data goes back to 1979 and is updated regularly. It is widely used in environmental and public health research.

### What does 'programmatic download' mean?

You could download gridMET files manually: go to a website, click a button, wait for the file, save it to your computer. But what if you need 46 years of data? Or data for multiple variables? Clicking through hundreds of downloads is slow, error-prone, and impossible to reproduce.

Programmatic downloading means writing code that fetches data automatically. You run your script, it downloads exactly what you specified, and anyone with the same script can reproduce your data retrieval step exactly.

### How we will download the data

gridMET data is hosted on a THREDDS server -- a type of data server designed for scientific datasets. The server supports a protocol called OPeNDAP, which lets you access a remote file as if it were open on your computer. You can even request just a spatial subset, so you only transfer the data you need.

The OPeNDAP URL for gridMET follows this pattern:

```
http://thredds.northwestknowledge.net:8080/thredds/dodsC/MET/{variable}/{variable}_{year}.nc
```

Where:
- `{variable}` is the short name for the climate variable (e.g., `tmmx` for max temperature)
- `{year}` is the four-digit year (e.g., `2023`)

We will open this URL with xarray, which handles the OPeNDAP protocol automatically. We will then select just the Louisiana region and save a small local copy (about 12 MB). Every section after this one will use that local file.

**Louisiana bounding box:**
- Latitude: 28.5 N to 33.5 N
- Longitude: 94.5 W to 88.5 W (stored as negative numbers: -94.5 to -88.5)

In [2]:
# --- Section 1: Download gridMET tmmx for Louisiana, 2023 ---

# OPeNDAP URL for gridMET maximum temperature, 2023
# xarray can open this URL directly -- it reads the file remotely over the network
opendap_url = (
    "http://thredds.northwestknowledge.net:8080/thredds/dodsC/"
    "MET/tmmx/tmmx_2023.nc"
)

# Path where we will save the local subset file
# We create the data/ directory if it does not already exist
os.makedirs("../data", exist_ok=True)
output_path = "../data/tmmx_2023_louisiana.nc"

print("Connecting to the gridMET THREDDS server...")
print(f"URL: {opendap_url}")

# Open the remote dataset via OPeNDAP
# At this point xarray reads only the metadata (variable names, dimensions, attributes)
# No actual temperature values are downloaded yet
ds_remote = xr.open_dataset(opendap_url, engine="netcdf4")
print(f"Connected! Full dataset dimensions: {dict(ds_remote.sizes)}")
print("  (That is the entire CONUS -- about 585 lat x 1386 lon x 365 days)")

# Subset to Louisiana using coordinate selection
# Note: lat values run from north (49.4) to south (25.1) -- descending order
# So we slice from the LARGER lat value to the SMALLER one: slice(33.5, 28.5)
print("Subsetting to Louisiana bounding box...")
ds_louisiana = ds_remote.sel(
    lat=slice(33.5, 28.5),   # Latitude: 33.5 N down to 28.5 N
    lon=slice(-94.5, -88.5)  # Longitude: 94.5 W to 88.5 W (negative = west)
)
print(f"Louisiana subset dimensions: {dict(ds_louisiana.sizes)}")

# Download all values into memory now
# This is the actual data transfer -- the ~12 MB download happens here
print("Downloading data into memory...")
ds_louisiana = ds_louisiana.load()

# Close the remote connection -- we no longer need the OPeNDAP link
ds_remote.close()
print("Download complete. Remote connection closed.")

# Remove existing file if present (avoids a file-lock PermissionError on re-run)
if os.path.exists(output_path):
    os.remove(output_path)

# Write the in-memory dataset to a local NetCDF file
print(f"Saving to {output_path}...")
ds_louisiana.to_netcdf(output_path)

# Report the file size
size_mb = os.path.getsize(output_path) / 1e6
print(f"Done! File saved. Size on disk: {size_mb:.1f} MB")

# Open the local file -- all subsequent sections will use this variable 'ds'
ds = xr.open_dataset(output_path)
print("Local file loaded successfully. Here is a quick summary:")
print(ds)

Connecting to the gridMET THREDDS server...
URL: http://thredds.northwestknowledge.net:8080/thredds/dodsC/MET/tmmx/tmmx_2023.nc
Connected! Full dataset dimensions: {'day': 365, 'lat': 585, 'lon': 1386, 'crs': 1}
  (That is the entire CONUS -- about 585 lat x 1386 lon x 365 days)
Subsetting to Louisiana bounding box...
Louisiana subset dimensions: {'day': 365, 'lat': 120, 'lon': 144, 'crs': 1}
Download complete. Remote connection closed.
Saving to ../data/tmmx_2023_louisiana.nc...
Done! File saved. Size on disk: 12.6 MB
Local file loaded successfully. Here is a quick summary:
<xarray.Dataset> Size: 50MB
Dimensions:          (day: 365, lat: 120, lon: 144, crs: 1)
Coordinates:
  * day              (day) datetime64[ns] 3kB 2023-01-01 ... 2023-12-31
  * lat              (lat) float64 960B 33.48 33.44 33.4 ... 28.61 28.57 28.52
  * lon              (lon) float64 1kB -94.47 -94.43 -94.39 ... -88.56 -88.52
  * crs              (crs) float32 4B 3.0
Data variables:
    air_temperature  (day, lat

### Reflection Questions -- Section 1

*Double-click this cell to edit it and type your answers.*

**Q1.** Look at the OPeNDAP URL structure explained above. What would you change in the URL if you wanted to download precipitation data (gridMET short name: `pr`) instead of maximum temperature?

*Your answer:*

---

**Q2.** What does the bounding box (lat 28.5-33.5, lon -94.5 to -88.5) represent, and why did we use one instead of downloading the entire CONUS?

*Your answer:*

---

**Q3.** Why is programmatic downloading better than manually downloading files one by one?

*Your answer:*

---
## Section 2: Understanding the Structure of a NetCDF File

### What is a NetCDF file?

NetCDF stands for **Network Common Data Form**. It is a file format designed specifically for storing scientific array data -- things like temperature fields that have three dimensions (time, latitude, longitude).

Think of it this way: a spreadsheet has rows and columns -- two dimensions. A NetCDF file is like a stack of spreadsheets, where each sheet is one day's worth of data, and every cell in the sheet holds the temperature at one geographic location on that day.

A NetCDF file has four main components:

| Component | What it is | Analogy |
|-----------|-----------|--------|
| **Dimensions** | The axes of the data (day, lat, lon) | Rows, columns, sheets in a workbook |
| **Variables** | The actual data arrays | The numbers inside the cells |
| **Coordinates** | The labeled tick marks along each axis | Row numbers and column headers |
| **Attributes** | Metadata (units, descriptions, source) | Notes written in a cell comment |

Let's explore each of these.

In [ ]:
# --- Section 2: Explore the NetCDF file structure ---

# Print the full xarray dataset summary
# This shows dimensions, coordinates, data variables, and global attributes
print("=== Full dataset summary ===")
print(ds)

In [ ]:
# --- List the data variables and their metadata ---

print("=== Data variables ===")
for var_name in ds.data_vars:
    # Access the DataArray for this variable
    var = ds[var_name]
    print(f"\nVariable: {var_name}")
    print(f"  Dimensions: {var.dims}")
    print(f"  Shape: {var.shape}")
    print(f"  Data type: {var.dtype}")
    # Print all attributes (units, description, etc.)
    print("  Attributes:")
    for attr_key, attr_val in var.attrs.items():
        print(f"    {attr_key}: {attr_val}")

In [ ]:
# --- Dimensions and their sizes ---

print("=== Dimensions and their sizes ===")
for dim_name, dim_size in ds.sizes.items():
    print(f"  {dim_name}: {dim_size} values")

# How many grid cells are in this dataset?
n_lat = ds.sizes['lat']   # Number of latitude steps
n_lon = ds.sizes['lon']   # Number of longitude steps
n_day = ds.sizes['day']   # Number of time steps
total_cells = n_lat * n_lon  # Spatial grid cells
total_values = n_lat * n_lon * n_day  # Total data values
print(f"\nSpatial grid cells (lat x lon): {n_lat} x {n_lon} = {total_cells:,}")
print(f"Total data values (lat x lon x day): {total_values:,}")

In [ ]:
# --- Access the temperature variable and inspect its shape ---

# The actual temperature variable is named 'air_temperature' in the NetCDF file
# 'tmmx' is gridMET's short name for it (visible in the long_name attribute)
tmmx = ds['air_temperature']

print("=== Temperature variable: air_temperature ===")
print(f"Shape: {tmmx.shape}")
print(f"This means: ({tmmx.shape[0]} days, {tmmx.shape[1]} lat cells, "
      f"{tmmx.shape[2]} lon cells)")
print(f"Units: {tmmx.attrs['units']}")
print(f"Description: {tmmx.attrs['description']}")

# Show a slice: temperature on the first day, as a 2D array
print(f"\nTemperature on January 1, 2023 (first 3 rows, first 5 cols, in Kelvin):")
print(tmmx.isel(day=0).values[:3, :5].round(2))
print("  (Kelvin = Celsius + 273.15, so ~300 K is about 27 C)")

### Reflection Questions -- Section 2

*Double-click this cell to edit it and type your answers.*

**Q1.** How many grid cells does this dataset have in total (latitude x longitude)? How did you figure that out from the output above?

*Your answer:*

---

**Q2.** What are the units of maximum temperature in this file? Where did you find that information?

*Your answer:*

---

**Q3.** What is the difference between a **dimension** and a **variable** in a NetCDF file? Use an analogy if it helps.

*Your answer:*

---
## Section 3: Understanding Spatial Resolution

### What is spatial resolution?

Spatial resolution describes how fine-grained your data is in space -- essentially, how small each data cell is.

Think of a digital photo. A high-resolution photo has many tiny pixels, so it looks sharp and detailed. A low-resolution photo has fewer, larger pixels and looks blurry. Gridded climate data works the same way: high resolution means small grid cells (more detail), low resolution means large grid cells (less detail).

**gridMET's resolution:** approximately 4 km. Each grid cell represents a roughly 4 km x 4 km square on the ground. For a state like Louisiana (about 560 km wide and 450 km tall), that means hundreds of grid cells in each direction.

Let's calculate the exact resolution and visualize what the grid looks like over Louisiana.

In [ ]:
# --- Calculate spatial resolution ---

# Extract the latitude and longitude coordinate arrays
lats = ds.lat.values   # 1D array of latitude values
lons = ds.lon.values   # 1D array of longitude values

print("=== Latitude coordinate ===")
print(f"  Number of lat values: {len(lats)}")
print(f"  Range: {lats[-1]:.3f} N to {lats[0]:.3f} N")
print(f"  Note: lat runs from NORTH to SOUTH (descending order)")

print("\n=== Longitude coordinate ===")
print(f"  Number of lon values: {len(lons)}")
print(f"  Range: {lons[0]:.3f} W to {lons[-1]:.3f} W")

# Calculate the spacing between adjacent grid cells (the resolution)
# abs() gives us the absolute value so it is always positive
dlat_deg = abs(float(lats[0]) - float(lats[1]))  # Degrees between lat cells
dlon_deg = abs(float(lons[0]) - float(lons[1]))  # Degrees between lon cells

# Convert degrees to approximate kilometers
# 1 degree of latitude = ~111 km everywhere
# 1 degree of longitude = ~111 km * cos(latitude) -- shrinks toward the poles
mid_lat = np.radians(31.0)  # Middle of Louisiana in radians
dlat_km = dlat_deg * 111.0
dlon_km = dlon_deg * 111.0 * np.cos(mid_lat)

print(f"\n=== Spatial resolution ===")
print(f"  Degrees: {dlat_deg:.5f} deg lat x {dlon_deg:.5f} deg lon")
print(f"  Kilometers (at 31 N): {dlat_km:.2f} km (N-S) x {dlon_km:.2f} km (E-W)")

print(f"\n=== Grid size over Louisiana ===")
print(f"  {len(lats)} lat cells x {len(lons)} lon cells = {len(lats)*len(lons):,} total grid cells")

In [ ]:
# --- Load Louisiana parish boundaries ---

# The US Census Bureau provides county/parish boundary shapefiles
# Louisiana parishes are equivalent to counties in other states (FIPS state code 22)
# We download the national county file and filter to Louisiana
census_url = (
    "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip"
)
print("Loading Louisiana parish boundaries from US Census TIGER/Line...")

# Read the shapefile directly from the URL (geopandas handles the download)
counties_national = gpd.read_file(census_url)

# Filter to Louisiana only (state FIPS code '22')
la_parishes = counties_national[counties_national['STATEFP'] == '22'].copy()

# Save the original CRS before reprojecting -- we will inspect it in Section 6
la_raw_crs = la_parishes.crs

# Reproject to WGS84 (EPSG:4326) to match the gridMET coordinate system
# We will discuss why this matters in Section 6
la_parishes = la_parishes.to_crs('EPSG:4326')

print(f"Number of Louisiana parishes: {len(la_parishes)}")
print(f"Original census CRS: {la_raw_crs}")
print(f"After reproject: {la_parishes.crs}")
print(f"Sample parish names: {la_parishes['NAME'].head(5).tolist()}")

In [ ]:
# --- Visualize the gridMET grid over Louisiana ---

# Create a meshgrid: every combination of lat and lon
# This gives us the center coordinates of all grid cells
lon_grid, lat_grid = np.meshgrid(lons, lats)
# lon_grid and lat_grid are both (120, 144) arrays

# Set up the figure with a Plate Carree (standard lat/lon) map projection
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

# Draw Louisiana parish boundaries from the shapefile
for geom in la_parishes.geometry:
    ax.add_geometries(
        [geom],
        crs=ccrs.PlateCarree(),
        facecolor='none',      # Transparent fill
        edgecolor='black',     # Black boundary lines
        linewidth=0.5
    )

# Plot every grid cell center as a small dot
# flatten() converts the 2D meshgrid arrays into 1D lists
ax.scatter(
    lon_grid.flatten(),     # X coordinates (longitude)
    lat_grid.flatten(),     # Y coordinates (latitude)
    s=1,                    # Dot size (very small so they don't overlap)
    color='steelblue',
    alpha=0.6,
    transform=ccrs.PlateCarree(),
    label='gridMET cell center'
)

# Set the map extent to our Louisiana bounding box
ax.set_extent([-94.5, -88.5, 28.5, 33.5], crs=ccrs.PlateCarree())

# Labels and formatting
ax.set_title(
    "gridMET Grid Cell Centers over Louisiana (2023, tmmx)",
    fontsize=13
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5)

# Add a legend patch for the parish boundaries
boundary_patch = mpatches.Patch(edgecolor='black', facecolor='none',
                                label='Parish boundaries')
cell_patch = mpatches.Patch(color='steelblue', label='gridMET cell centers')
ax.legend(handles=[boundary_patch, cell_patch], loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()
print(f"Each blue dot represents one ~4 km x 4 km gridMET cell.")
print(f"Total dots: {lon_grid.size:,}")

### Reflection Questions -- Section 3

*Double-click this cell to edit it and type your answers.*

**Q1.** If gridMET had 10 km resolution instead of 4 km, would there be more or fewer grid cells over Louisiana? Why?

*Your answer:*

---

**Q2.** Why might a researcher care about the spatial resolution of their dataset?

*Your answer:*

---

**Q3.** Looking at the map, approximately how many grid cell centers fall within a single Louisiana parish? (Pick a medium-sized parish and eyeball it.)

*Your answer:*

---
## Section 4: Understanding Temporal Resolution

### What is temporal resolution?

Just as spatial resolution describes how fine-grained the data is in space, **temporal resolution** describes how fine-grained it is in time.

gridMET is a **daily** dataset -- one snapshot per day per grid cell. Other datasets might be hourly, monthly, or annual. Daily data lets us capture short-term events (a single extreme heat day, a storm, a cold snap) that monthly averages would smooth over.

### How is time stored in NetCDF files?

Raw NetCDF files often store time as an integer: the number of days since some reference date (for example, 'days since 1900-01-01'). A value of 44927, for instance, means 44,927 days after January 1, 1900.

xarray automatically decodes these integers into human-readable dates (like `2023-01-01`), so you rarely have to deal with the raw encoding. But it is useful to know the encoding exists -- you will see references to 'days since ...' in NetCDF metadata.

### The time dimension in our file

Notice that the time dimension in this file is called **`day`** (not `time`). Variable and dimension names are chosen by the dataset creator, and they vary across datasets. Always inspect the dataset structure (as we did in Section 2) before writing analysis code.

In [ ]:
# --- Explore the time coordinate ---

# Access the 'day' coordinate (the time dimension in this file)
time_coord = ds['day']

print("=== Time coordinate ===")
print(f"Name: 'day'  (note: some datasets use 'time' instead)")
print(f"Number of time steps: {len(time_coord)}")
print(f"First date: {str(time_coord.values[0])[:10]}")
print(f"Last date:  {str(time_coord.values[-1])[:10]}")

# The underlying encoding (how time is stored before xarray decodes it)
print(f"\nTime encoding: {time_coord.attrs.get('description', 'see units')}")

print("\nFirst 10 dates:")
for t in time_coord.values[:10]:
    # Convert numpy datetime64 to a readable string
    print(f"  {str(t)[:10]}")

In [ ]:
# --- Slice the data for specific time periods ---

# Slicing a single day
# xarray's .sel() method selects by label (the actual date)
single_day = ds['air_temperature'].sel(day='2023-07-04')
print(f"Single day (July 4, 2023) shape: {single_day.shape}")
print(f"  (Just lat x lon -- time dimension is gone after selecting one day)")

# Slicing a full month using a string prefix
# '2023-07' matches all days in July 2023
july = ds['air_temperature'].sel(day='2023-07')
print(f"\nJuly 2023 shape: {july.shape}")
print(f"  ({july.shape[0]} days x {july.shape[1]} lat x {july.shape[2]} lon)")

# Slicing a season (June, July, August = summer)
summer = ds['air_temperature'].sel(day=slice('2023-06-01', '2023-08-31'))
print(f"\nSummer 2023 (Jun-Aug) shape: {summer.shape}")
print(f"  ({summer.shape[0]} days in summer)")

# Winter (January and February 2023)
winter = ds['air_temperature'].sel(day=slice('2023-01-01', '2023-02-28'))
print(f"\nEarly winter 2023 (Jan-Feb) shape: {winter.shape}")

In [ ]:
# --- Time series plot: max temperature near Baton Rouge for all of 2023 ---

# Baton Rouge coordinates: 30.45 N, 91.15 W
# We use .sel() with method='nearest' to find the closest grid cell
baton_rouge_lat = 30.45
baton_rouge_lon = -91.15  # Negative because west of the prime meridian

# Select the grid cell nearest to Baton Rouge and get all 365 days
ts_kelvin = ds['air_temperature'].sel(
    lat=baton_rouge_lat,
    lon=baton_rouge_lon,
    method='nearest'  # Find the closest lat/lon values in the grid
)

# Convert from Kelvin to Celsius for a more intuitive scale
ts_celsius = ts_kelvin - 273.15

# Get the actual grid cell coordinates that were selected
actual_lat = float(ts_kelvin.lat)
actual_lon = float(ts_kelvin.lon)
print(f"Requested: {baton_rouge_lat} N, {baton_rouge_lon} W")
print(f"Nearest grid cell: {actual_lat:.3f} N, {actual_lon:.3f} W")
print(f"Time series shape: {ts_celsius.shape} (one value per day)")

# --- Build the plot ---
fig, ax = plt.subplots(figsize=(14, 5))

# Plot the daily time series as a line
ax.plot(ts_celsius.day.values, ts_celsius.values, color='firebrick',
        linewidth=0.8, label='Daily max temp')

# Add seasonal shading to orient the reader
# Winter: Jan 1 - Mar 20
ax.axvspan(np.datetime64('2023-01-01'), np.datetime64('2023-03-20'),
           alpha=0.08, color='steelblue', label='Winter')
# Spring: Mar 20 - Jun 21
ax.axvspan(np.datetime64('2023-03-20'), np.datetime64('2023-06-21'),
           alpha=0.08, color='lightgreen', label='Spring')
# Summer: Jun 21 - Sep 22
ax.axvspan(np.datetime64('2023-06-21'), np.datetime64('2023-09-22'),
           alpha=0.08, color='orange', label='Summer')
# Fall: Sep 22 - Dec 21
ax.axvspan(np.datetime64('2023-09-22'), np.datetime64('2023-12-21'),
           alpha=0.08, color='peru', label='Fall')
# Late winter: Dec 21 - Dec 31
ax.axvspan(np.datetime64('2023-12-21'), np.datetime64('2023-12-31'),
           alpha=0.08, color='steelblue')

# Add a horizontal reference line at 37.8 C (100 F)
ax.axhline(37.8, color='darkred', linestyle='--', linewidth=1,
           label='100 F (37.8 C)')

# Labels and formatting
ax.set_title(
    f"Daily Max Temperature near Baton Rouge, LA (2023)\n"
    f"gridMET cell: {actual_lat:.3f} N, {actual_lon:.3f} W",
    fontsize=12
)
ax.set_xlabel("Date")
ax.set_ylabel("Max Temperature (Celsius)")
ax.legend(loc='upper right', fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Annual min: {float(ts_celsius.min()):.1f} C")
print(f"Annual max: {float(ts_celsius.max()):.1f} C")
print(f"Annual mean: {float(ts_celsius.mean()):.1f} C")

### Reflection Questions -- Section 4

*Double-click this cell to edit it and type your answers.*

**Q1.** How many time steps are in the 2023 dataset? Does that match what you expected for a daily dataset?

*Your answer:*

---

**Q2.** Looking at the time series plot, in which month was max temperature highest near Baton Rouge? Does that match your expectation for Louisiana?

*Your answer:*

---

**Q3.** If you wanted to study heat waves or drought, why would daily data be more useful than monthly data?

*Your answer:*

---
## Section 5: Visualizing Gridded Data

### What does it mean to visualize gridded data?

So far we have looked at the data as numbers in arrays and as a time series at a single location. Now we will make a **map** -- a spatial view of the data where color represents the value at each grid cell.

This type of visualization is called a **raster map**. 'Raster' means the data is organized as a regular grid of cells, where each cell has exactly one value. Think of it like coloring in a spreadsheet: each row-column combination gets a color based on the number in that cell.

### Colormaps

A **colormap** is the mapping from data values to colors. If you are showing temperature, a common choice is a colormap that goes from cool blue (low temp) to warm red (high temp). The choice of colormap matters: a poorly chosen colormap can make differences invisible or create visual artifacts. For temperature data, sequential colormaps (one hue getting lighter or darker) or diverging colormaps (two contrasting hues meeting in the middle) both work well.

We will compare a hot summer day and a cooler winter day side by side.

In [ ]:
# --- Select data for two contrasting days ---

# August 15, 2023 -- a hot summer day in Louisiana
summer_day = ds['air_temperature'].sel(day='2023-08-15')
# January 15, 2023 -- a cool winter day in Louisiana
winter_day = ds['air_temperature'].sel(day='2023-01-15')

# Convert from Kelvin to Celsius for both days
summer_C = summer_day - 273.15
winter_C = winter_day - 273.15

print(f"August 15 -- min: {float(summer_C.min()):.1f} C, "
      f"max: {float(summer_C.max()):.1f} C, "
      f"mean: {float(summer_C.mean()):.1f} C")
print(f"January 15 -- min: {float(winter_C.min()):.1f} C, "
      f"max: {float(winter_C.max()):.1f} C, "
      f"mean: {float(winter_C.mean()):.1f} C")

# Use a shared color scale so both maps are directly comparable
# We find the overall min and max across both days
vmin = min(float(winter_C.min()), float(summer_C.min()))
vmax = max(float(winter_C.max()), float(summer_C.max()))
print(f"\nShared color scale: {vmin:.1f} to {vmax:.1f} C")

In [ ]:
# --- Plot summer and winter maps side by side ---

fig, axes = plt.subplots(
    1, 2,                         # 1 row, 2 columns
    figsize=(16, 7),
    subplot_kw={'projection': ccrs.PlateCarree()}  # Map projection
)

titles = ['August 15, 2023 (summer)', 'January 15, 2023 (winter)']
datasets = [summer_C, winter_C]

# We will store the last image object so we can attach one colorbar to both maps
img = None

for ax, title, data in zip(axes, titles, datasets):
    # pcolormesh draws a filled color grid
    # lon (X) and lat (Y) define the grid edges; data.values provides the colors
    img = ax.pcolormesh(
        ds.lon.values,   # X coordinates (longitude)
        ds.lat.values,   # Y coordinates (latitude)
        data.values,     # Color values (temperature in Celsius)
        cmap='RdYlBu_r', # Colormap: red=hot, blue=cool
        vmin=vmin,       # Shared color scale minimum
        vmax=vmax,       # Shared color scale maximum
        transform=ccrs.PlateCarree()
    )

    # Draw parish boundaries on top of the color grid
    for geom in la_parishes.geometry:
        ax.add_geometries(
            [geom],
            crs=ccrs.PlateCarree(),
            facecolor='none',
            edgecolor='black',
            linewidth=0.4
        )

    # Set map extent, labels, and gridlines
    ax.set_extent([-94.5, -88.5, 28.5, 33.5], crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.4)

# Add a single shared colorbar below both maps
cbar = fig.colorbar(img, ax=axes, orientation='horizontal',
                    pad=0.08, fraction=0.04, shrink=0.7)
cbar.set_label('Daily Max Temperature (Celsius)', fontsize=11)

fig.suptitle(
    "gridMET Maximum Temperature over Louisiana",
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.show()

### Reflection Questions -- Section 5

*Double-click this cell to edit it and type your answers.*

**Q1.** What do the colors on the map represent? How would you change the colormap to make the hottest areas even more visually obvious? (Hint: look up matplotlib colormap options.)

*Your answer:*

---

**Q2.** What spatial patterns do you notice in the summer map -- are some parts of Louisiana hotter than others? What might explain that?

*Your answer:*

---

**Q3.** Why do we overlay parish boundaries on top of the gridded data? What does it help you understand?

*Your answer:*

---
## Section 6: Coordinate Reference Systems and Projections

### What is a Coordinate Reference System (CRS)?

The Earth is roughly a sphere, but maps are flat. To go from a sphere to a flat surface, you need a **projection** -- a mathematical transformation that 'peels' the globe onto a plane. Different projections make different trade-offs: some preserve shape, some preserve area, some preserve distances. None can preserve everything perfectly.

A **Coordinate Reference System (CRS)** defines:
1. How the Earth's shape is modeled (the 'datum' -- is it a perfect sphere, or an ellipsoid that bulges at the equator?)
2. How coordinates (latitude/longitude, or x/y in meters) map to locations on that model

### WGS84 and NAD83

The most common geographic CRS is **WGS84** (World Geodetic System 1984, EPSG:4326). It is the standard used by GPS devices worldwide. Coordinates are in degrees of latitude and longitude.

**NAD83** (North American Datum 1983, EPSG:4269) is a datum optimized for North America. For most practical purposes, WGS84 and NAD83 are nearly identical -- they differ by less than 1 meter in most locations -- but they are technically different CRSes, and software treats them as such. This is exactly what we found in this project: the Census shapefile came in NAD83, so we reprojected it to WGS84 before overlaying it on the gridMET data.

### Why CRS matters when combining datasets

If you overlay a shapefile in one CRS on top of a raster in a different CRS without first reprojecting, the features will not line up correctly. This can produce subtle or dramatic misalignments depending on how different the two CRSes are. Always check the CRS of every dataset you combine.

Let's check the CRSes of our two datasets.

In [ ]:
# --- CRS of the gridMET data ---

# gridMET stores CRS information in a special 'crs' variable and global attributes
print("=== gridMET CRS information ===")

# Check the global attribute
gridmet_crs_epsg = ds.attrs.get('geospatial_bounds_crs', 'not in global attrs')
print(f"  Global attribute 'geospatial_bounds_crs': {gridmet_crs_epsg}")

# The crs variable contains detailed projection metadata
crs_var = ds['crs']
print(f"  CRS variable 'long_name': {crs_var.attrs.get('long_name', 'n/a')}")
print(f"  Coordinate system: {ds['air_temperature'].attrs.get('coordinate_system', 'n/a')}")

print("\nConclusion: gridMET uses WGS84 (EPSG:4326) -- standard lat/lon degrees")

In [ ]:
# --- CRS of the Louisiana parish shapefile ---

# la_parishes was loaded from the Census TIGER/Line file in Section 3
# We saved the original (pre-reproject) CRS as la_raw_crs in that section
# No need to re-download the 83 MB national file -- we already have this info

print("=== Census TIGER/Line shapefile CRS ===")
print(f"Original CRS (before reprojection): {la_raw_crs}")
print(f"  EPSG code: {la_raw_crs.to_epsg()}")
print(f"  This is NAD83 -- nearly identical to WGS84 but technically different")

print(f"\nAfter reprojection (done in Section 3):")
print(f"  la_parishes CRS: {la_parishes.crs}")
print(f"  EPSG code: {la_parishes.crs.to_epsg()}")
print(f"  Now both datasets are in WGS84 (EPSG:4326)")

print("\n=== Reprojection command we used ===")
print("  la_parishes = la_parishes.to_crs('EPSG:4326')")
print("  .to_crs() is a one-line command that converts between any two CRSes")

### A note on what would happen without reprojection

For WGS84 and NAD83, the difference is tiny (< 1 meter), so in practice overlaying them without reprojection would look fine. But consider a more extreme case: if one dataset used a metric projection in meters (like UTM, where coordinates might be 500,000, 3,000,000) and you tried to overlay it on a lat/lon dataset (where coordinates are -91, 30), the software would try to find coordinates 500,000 degrees east and 3,000,000 degrees north -- completely off the map. Always check CRS before combining datasets.

### Reflection Questions -- Section 6

*Double-click this cell to edit it and type your answers.*

**Q1.** In your own words, what is a CRS and why does it matter for combining spatial datasets?

*Your answer:*

---

**Q2.** What would happen if you tried to overlay two datasets with very different CRSes (e.g., one in degrees lat/lon and one in meters) without reprojecting?

*Your answer:*

---

**Q3.** Why do you think WGS84 (EPSG:4326) is one of the most commonly used CRSes for environmental data?

*Your answer:*

---
## Section 7: Bridge to Parish-Level Analysis

### From grid cells to administrative units

Everything we have done so far has worked at the level of individual grid cells -- 4 km x 4 km squares. But in public health research, we often want to link climate exposure to health outcomes at the scale of an administrative unit: a county, a parish, a census tract.

Health records, hospital data, and vital statistics are reported at these administrative boundaries, not at individual grid cells. To link climate data to health data, we need to **aggregate** grid cell values up to the administrative unit level.

### What is spatial aggregation?

Spatial aggregation means summarizing many values from a fine-grained source (the grid) into a single value for a coarser unit (the parish). The most common approach is to:

1. Identify which grid cells fall **inside** the parish boundary
2. Take the **average** of those grid cell values

Step 1 is called a **point-in-polygon** operation -- for each grid cell center (a point), we test whether it falls inside the parish polygon. This is exactly what we will do below.

### What we are building toward

The research project that this notebook supports will aggregate gridMET temperature data to all 64 Louisiana parishes for every day from 1979 to 2025. That is 64 parishes x ~17,000 days = over 1 million parish-day values. In this section, we will do the same operation for one parish on one day -- so you understand exactly what the pipeline does at full scale.

In [ ]:
# --- Select East Baton Rouge Parish ---

# la_parishes is already loaded and projected to WGS84
ebr = la_parishes[la_parishes['NAME'] == 'East Baton Rouge'].copy()
print(f"East Baton Rouge Parish")
print(f"  GEOID: {ebr['GEOID'].values[0]}")
print(f"  CRS: {ebr.crs}")
print(f"  Bounding box: {ebr.total_bounds.round(3)}")

# Extract the polygon geometry for point-in-polygon testing
ebr_geom = ebr.geometry.iloc[0]

In [ ]:
# --- Find which grid cell centers fall inside East Baton Rouge Parish ---

# We already have a meshgrid of all lat/lon combinations from Section 3
# (lon_grid and lat_grid, both shape 120 x 144)

print("Testing each grid cell center against the EBR parish boundary...")
print(f"Total grid cells to test: {lon_grid.size:,}")

# Build a boolean mask: True where the grid cell center is inside EBR
inside_ebr = np.zeros((len(lats), len(lons)), dtype=bool)

for i, lat_val in enumerate(lats):
    for j, lon_val in enumerate(lons):
        # Create a Shapely Point at this grid cell center
        pt = Point(lon_val, lat_val)
        # Test if the point is inside the EBR polygon
        inside_ebr[i, j] = ebr_geom.contains(pt)

n_cells_ebr = inside_ebr.sum()
print(f"\nGrid cells inside East Baton Rouge Parish: {n_cells_ebr}")
print(f"This means roughly {n_cells_ebr} x (4km x 4km) = "
      f"{n_cells_ebr * 16:.0f} sq km of grid coverage")

In [ ]:
# --- Map: grid cells over all of Louisiana, with EBR highlighted ---

fig, ax = plt.subplots(
    figsize=(10, 9),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

# Draw all Louisiana parish boundaries (gray)
for geom in la_parishes.geometry:
    ax.add_geometries(
        [geom], crs=ccrs.PlateCarree(),
        facecolor='none', edgecolor='gray', linewidth=0.4
    )

# Highlight East Baton Rouge Parish with a yellow fill
ax.add_geometries(
    [ebr_geom], crs=ccrs.PlateCarree(),
    facecolor='lightyellow', edgecolor='black', linewidth=1.5
)

# Plot all grid cell centers over Louisiana (gray, small)
ax.scatter(
    lon_grid.flatten(), lat_grid.flatten(),
    s=1, color='lightgray', alpha=0.8,
    transform=ccrs.PlateCarree(), label='All grid cells'
)

# Highlight the cells that fall inside EBR (orange, larger)
ax.scatter(
    lon_grid[inside_ebr],   # Only the lon values where inside_ebr is True
    lat_grid[inside_ebr],   # Matching lat values
    s=12, color='darkorange', zorder=5,
    transform=ccrs.PlateCarree(),
    label=f'Inside EBR ({n_cells_ebr} cells)'
)

# Show full Louisiana extent
ax.set_extent([-94.5, -88.5, 28.5, 33.5], crs=ccrs.PlateCarree())

ax.set_title(
    "gridMET Grid Cells Across Louisiana\n"
    "Orange = cells inside East Baton Rouge Parish",
    fontsize=13
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.4)
ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Calculate the mean max temperature in EBR for one day ---

# Select August 15, 2023
date = '2023-08-15'
day_data_K = ds['air_temperature'].sel(day=date).values  # Shape: (120, 144)

# Use the inside_ebr mask to extract values for EBR cells only
ebr_cell_temps_K = day_data_K[inside_ebr]  # 1D array of EBR cell temperatures
print(f"Date: {date}")
print(f"Number of EBR grid cells: {len(ebr_cell_temps_K)}")
print(f"Individual cell temperatures (K): {ebr_cell_temps_K.round(2)}")

# Convert to Celsius and compute the mean
ebr_cell_temps_C = ebr_cell_temps_K - 273.15
ebr_mean_C = float(ebr_cell_temps_C.mean())
ebr_min_C = float(ebr_cell_temps_C.min())
ebr_max_C = float(ebr_cell_temps_C.max())

print(f"\nSummary across EBR grid cells on {date}:")
print(f"  Cell-level min:  {ebr_min_C:.2f} C")
print(f"  Cell-level max:  {ebr_max_C:.2f} C")
print(f"  Cell-level mean: {ebr_mean_C:.2f} C  <-- this is the parish-day value")
print(f"\nThe single parish-day temperature for EBR on {date}: {ebr_mean_C:.2f} C")

### What we just did -- and what comes next

You just computed a single parish-day temperature value:

1. We found all gridMET cells whose centers fall inside East Baton Rouge Parish
2. We extracted their temperature values for August 15, 2023
3. We took the mean across those cells

That mean is the number we would use as 'the temperature in EBR on that day' for a health analysis.

In the actual research project, we will scale this process to **all 64 Louisiana parishes** and **all days from 1979 to 2025** -- roughly 17,000 days. The logic is exactly what you just did here: find the cells in the parish, extract the values, take the mean. The research pipeline automates this loop over all parishes and all days.

---

### Reflection Questions -- Section 7

*Double-click this cell to edit it and type your answers.*

**Q1.** Why do we take the **mean** of grid cells within a parish rather than just picking the one grid cell closest to the parish center?

*Your answer:*

---

**Q2.** What practical challenges do you think might arise when doing this aggregation for all 64 parishes and 17,000 days?

*Your answer:*

---

**Q3.** Based on everything you have learned in this notebook, describe in your own words the full pipeline: starting from the gridMET server, ending with a single daily temperature value for each Louisiana parish.

*Your answer:*